# Bài 6(c) — So sánh thời gian Apriori và FP-Growth

**Võ Thiện Linh — MSSV: 6451071041**

Notebook này tự chạy lại cả hai thuật toán trên `groceries.csv` để phép so sánh không phụ thuộc vào trạng thái hoặc thứ tự chạy notebook khác. `min_sup` đi từ 0,01 đến 0,1 theo tỉ lệ (1%–10%), tăng 0,01; thời gian là trung vị của ba lần chạy.

## Đọc dữ liệu và thiết lập

In [ ]:
import csv
import math
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from statistics import median

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

data_candidates = [Path('groceries.csv'), Path('6451071041_LAB1_Bai6_Trang45') / 'groceries.csv', Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai6_Trang45' / 'groceries.csv']
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy groceries.csv. Thư mục đang chạy: {Path.cwd()}')
with data_path.open(encoding='utf-8', newline='') as csv_file:
    transactions = [frozenset(item.strip() for item in row if item.strip()) for row in csv.reader(csv_file)]
transaction_count = len(transactions)
item_count = len(set().union(*transactions))
print(f'Đã đọc: {data_path.resolve()}')
print(f'Số giao dịch: {transaction_count:,}; số mặt hàng: {item_count:,}')
assert (transaction_count, item_count) == (9835, 169)
support_levels = [percent / 100 for percent in range(1, 11)]
repeat_count = 3

## Cài đặt các thuật toán để so sánh
Cả hai thuật toán xử lý cùng các giao dịch, cùng cách quy đổi ngưỡng hỗ trợ sang số giao dịch tối thiểu và cùng được tính thời gian bằng `perf_counter`.

In [ ]:
def apriori(transactions, min_count):
    item_counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in item_counts.items() if count >= min_count}
    all_frequent = dict(previous)
    frequent_singletons = set(item_counts)
    itemset_size = 2
    while previous:
        previous_tuples = sorted(tuple(sorted(itemset)) for itemset in previous)
        previous_set = set(previous)
        candidates = set()
        for left, right in combinations(previous_tuples, 2):
            if left[:-1] != right[:-1]:
                continue
            candidate = frozenset(left + (right[-1],))
            if len(candidate) == itemset_size and all(frozenset(subset) in previous_set for subset in combinations(candidate, itemset_size - 1)):
                candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            active_items = sorted(transaction & frequent_singletons)
            if len(active_items) >= itemset_size:
                for parts in combinations(active_items, itemset_size):
                    itemset = frozenset(parts)
                    if itemset in candidates:
                        candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= min_count}
        all_frequent.update(previous)
        itemset_size += 1
    return all_frequent

class FPNode:
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

def fp_growth(transactions, min_count):
    def mine(weighted_transactions, suffix=frozenset()):
        item_counts = Counter()
        for items, weight in weighted_transactions:
            item_counts.update({item: weight for item in set(items)})
        frequent_items = {item: count for item, count in item_counts.items() if count >= min_count}
        if not frequent_items:
            return {}
        root = FPNode(None)
        headers = {item: [] for item in frequent_items}
        for items, weight in weighted_transactions:
            ordered_items = sorted((item for item in set(items) if item in frequent_items), key=lambda item: (-frequent_items[item], item))
            current = root
            for item in ordered_items:
                child = current.children.get(item)
                if child is None:
                    child = FPNode(item, current)
                    current.children[item] = child
                    headers[item].append(child)
                child.count += weight
                current = child
        patterns = {}
        for item in sorted(headers, key=lambda value: (frequent_items[value], value)):
            pattern = suffix | frozenset([item])
            patterns[pattern] = frequent_items[item]
            conditional_paths = []
            for node in headers[item]:
                path = []
                parent = node.parent
                while parent is not None and parent.item is not None:
                    path.append(parent.item)
                    parent = parent.parent
                if path:
                    conditional_paths.append((path, node.count))
            if conditional_paths:
                patterns.update(mine(conditional_paths, pattern))
        return patterns
    return mine([(transaction, 1) for transaction in transactions])

## Kết quả, bảng tập phổ biến và biểu đồ

In [ ]:
apriori_results, fp_results = {}, {}
apriori_times, fp_times = {}, {}
rows, itemset_rows = [], []
for min_sup in support_levels:
    min_count = math.ceil(min_sup * transaction_count)
    elapsed_apriori, elapsed_fp = [], []
    for _ in range(repeat_count):
        start = time.perf_counter()
        apriori_result = apriori(transactions, min_count)
        elapsed_apriori.append(time.perf_counter() - start)
        start = time.perf_counter()
        fp_result = fp_growth(transactions, min_count)
        elapsed_fp.append(time.perf_counter() - start)
    assert apriori_result == fp_result, f'Kết quả không khớp tại min_sup={min_sup:.0%}'
    apriori_results[min_sup], fp_results[min_sup] = apriori_result, fp_result
    apriori_times[min_sup], fp_times[min_sup] = median(elapsed_apriori), median(elapsed_fp)
    rows.append({'min_sup': f'{min_sup:.0%}', 'min_count': min_count, 'Số tập phổ biến': len(apriori_result), 'Apriori (giây)': apriori_times[min_sup], 'FP-Growth (giây)': fp_times[min_sup], 'Tỉ lệ thời gian Apriori / FP-Growth': apriori_times[min_sup] / fp_times[min_sup]})
    for itemset, count in sorted(apriori_result.items(), key=lambda pair: (len(pair[0]), tuple(sorted(pair[0])))):
        itemset_rows.append({'min_sup': f'{min_sup:.0%}', 'tập mặt hàng': ', '.join(sorted(itemset)), 'kích thước tập': len(itemset), 'số giao dịch hỗ trợ': count, 'support thực tế': count / transaction_count})
comparison = pd.DataFrame(rows)
all_frequent_itemsets = pd.DataFrame(itemset_rows)
print('So sánh thời gian (trung vị của 3 lần chạy):')
display(comparison)
print(f'Tổng số tập phổ biến qua tất cả các mức: {len(all_frequent_itemsets):,}')
display(all_frequent_itemsets)
fig, ax = plt.subplots(figsize=(9, 5))
x = [support * 100 for support in support_levels]
ax.plot(x, [apriori_times[s] for s in support_levels], marker='o', linewidth=2, color='#0072B2', label='Apriori')
ax.plot(x, [fp_times[s] for s in support_levels], marker='s', linewidth=2, color='#E69F00', label='FP-Growth')
ax.set_title('So sánh thời gian tìm tập phổ biến theo min_sup')
ax.set_xlabel('min_sup (%)')
ax.set_ylabel('Thời gian xử lý (giây)')
ax.set_xticks(x, [f'{support:.0%}' for support in support_levels])
ax.set_ylim(bottom=0)
ax.grid(axis='y', linestyle='--', color='#D9D9D9', alpha=0.7)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
mean_apriori = comparison['Apriori (giây)'].mean()
mean_fp = comparison['FP-Growth (giây)'].mean()
fastest = 'FP-Growth' if mean_fp < mean_apriori else 'Apriori'
speed_ratio = max(mean_apriori, mean_fp) / min(mean_apriori, mean_fp)
print(f'Hai thuật toán cho cùng tập phổ biến ở cả {len(support_levels)} mức min_sup.')
print(f'Thời gian trung bình: Apriori = {mean_apriori:.4f} giây; FP-Growth = {mean_fp:.4f} giây.')
print(f'Trong lần chạy này, {fastest} nhanh hơn trung bình khoảng {speed_ratio:.2f} lần.')
print('Nhận xét: giảm min_sup thường làm tăng số mẫu phải khai thác. Apriori sinh và kiểm tra ứng viên theo từng tầng; FP-Growth nén giao dịch vào cây và khai thác các mẫu điều kiện, giảm chi phí sinh ứng viên. Thời gian thực tế phụ thuộc máy và cách cài đặt.')